# Final Assessment & Independent Exercise: Multi-Sensor Alternative Spectral Index & Time-Series Export

## Instrucciones

### Create a new Jupyter Notebook in your GitHub repository and replicate what you consider is necessary to reach what is required as follows:

1. Choose one index distinct from NDVI (e.g., EVI, SAVI, or NDWI) for canopy, soil, or moisture evaluation.
2. Generate annual median composites for both Sentinel-2 and Landsat 8 over your area, applying sensor-specific cloud masking (SCL / QA_PIXEL) and Landsat 8 Surface Reflectance scaling factors.
3. Construct Multi-Temporal Stacks: Combine the 10 annual index layers (2016 to 2025) into a single 10-band ee.Image stack for Sentinel-2 and a corresponding 10-band stack for Landsat 8 (renaming bands sequentially as Index_2016, Index_2017, ..., Index_2025).
4. GeoTIFF Export: Export both 10-band image stacks to Google Drive as projected GeoTIFF rasters using the official local CRS for Colombia (EPSG:9377).
5. Commit and push your final .ipynb file to your public GitHub repository.
6. Ensure your notebook contains executed output cells, structured Markdown headers, and code comments explaining your custom study area selection.
7. Copy the direct link to your Jupyter Notebook on GitHub and submit it using the following link

## Importar librerias 

In [7]:
import ee
import geopandas as gpd
from pathlib import Path
import geemap

## Autenticación GEE

In [8]:
ee.Authenticate()

Enter verification code:  4/1ATsMZqBd_IQ4KIMZQ6nuu2JUkoTP8pJfr7SBehY0RfPB0cV4eqYP5N9SEhM



Successfully saved authorization token.


### Instalación de la API GEE

In [9]:

try:
    ee.Initialize()
    print("Google Earth Engine initialised successfully.")
except Exception as e:
    print(f"Error initialising GEE: {e}")

Google Earth Engine initialised successfully.


C:\Users\Leo\miniconda3\envs\Geoprocesamiento\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


## Selección de área de estudio

In [13]:
root_folder=Path(r"C:\Users\Leo\Documents\GEOPROCESAMIENTO")
# Carcar los municipios de Colombia
gdf = gpd.read_file(root_folder /"municipios_colombia.gpkg")

# verificar el sistema de coordenadas
print(f"Initial CRS: {gdf.crs}")

# Reproyectar al sistema EPSG:9377
gdf_unico = gdf.to_crs(epsg=9377)
#revisar los datos
gdf_unico.head()

Initial CRS: EPSG:3116


,DPTO_CCDGO,MPIO_CCDGO,MPIO_CNMBR,MPIO_CDPMP,VERSION,AREA,LATITUD,LONGITUD,STCTNENCUE,STP3_1_SI,...,STP51_PRIM,STP51_SECU,STP51_SUPE,STP51_POST,STP51_13_E,STP51_99_E,Shape_Leng,Shape_Area,Codigo_Mun,geometry
0,18,001,FLORENCIA,18001,2018,2.547638e+09,1.749139,-75.558239,71877.0,32.0,...,48848.0,59610.0,21898.0,4592.0,5892.0,3799.0,2.942508,0.206928,18001,"MULTIPOLYGON (((4730856.146 1800689.038, 47308..."
1,18,029,ALBANIA,18029,2018,4.141221e+08,1.227865,-75.882327,2825.0,24.0,...,1940.0,1712.0,231.0,41.0,215.0,46.0,1.112829,0.033618,18029,"MULTIPOLYGON (((4677933.827 1709133.846, 46779..."
2,18,094,BELÉN DE LOS ANDAQUÍES,18094,2018,1.191619e+09,1.500923,-75.875645,4243.0,54.0,...,3541.0,3340.0,490.0,119.0,720.0,123.0,2.234657,0.096745,18094,"MULTIPOLYGON (((4690015.614 1751610.86, 469000..."
3,18,247,EL DONCELLO,18247,2018,1.106076e+09,1.791386,-75.193944,8809.0,0.0,...,7571.0,6287.0,1029.0,228.0,1095.0,171.0,3.154370,0.089867,18247,"MULTIPOLYGON (((4737450.122 1814755.048, 47374..."
4,18,256,EL PAUJÍL,18256,2018,1.234734e+09,1.617746,-75.234043,5795.0,0.0,...,6072.0,4066.0,639.0,108.0,916.0,99.0,3.529316,0.100309,18256,"MULTIPOLYGON (((4736905.653 1802381.382, 47376..."


In [14]:
# Calcular área en km² (ya reproyectado a EPSG:9377, en metros)
gdf_unico["area_km2"] = gdf_unico.geometry.area / 1_000_000

# Encontrar el municipio con menor área
municipio_mas_pequeno = gdf_unico.loc[gdf_unico["area_km2"].idxmin()]

print(f"Municipio con menor área: {municipio_mas_pequeno['MPIO_CNMBR']}")
print(f"Área: {municipio_mas_pequeno['area_km2']:.2f} km²")

Municipio con menor área: SABANETA
Área: 15.83 km²


## Filtrar por municipio y seleccionar SABANETA

In [15]:
# Filtrar el municipio de Sabaneta
gdf_muni = gdf_unico[gdf_unico["MPIO_CNMBR"] == "SABANETA"].copy()

# Verificar que se seleccionó correctamente
print(gdf_muni[["MPIO_CNMBR", "area_km2"]])

     MPIO_CNMBR   area_km2
1120   SABANETA  15.831397


## Convertir la geometría del municipioa formato Earth Engine

In [19]:
bbox = gdf_muni.to_crs(epsg=4326).total_bounds
ee_bounds = ee.Geometry.BBox(bbox[0], bbox[1], bbox[2], bbox[3])

geojson_geom = gdf_muni.to_crs(epsg=4326).geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

# Definir las funciones de máscara de nubes (S2 y L8)

In [17]:
def mask_s2_clouds_scl(image):
    """Masks clouds, cloud shadows, and cirrus using the SCL band."""
    scl = image.select('SCL')
    cloud_shadows = scl.eq(3)
    clouds_medium = scl.eq(8)
    clouds_high = scl.eq(9)
    cirrus = scl.eq(10)
    mask = cloud_shadows.Or(clouds_medium).Or(clouds_high).Or(cirrus).Not()
    return image.updateMask(mask)

def mask_l8_clouds_qa(image):
    """Masks clouds and cloud shadows using QA_PIXEL band for Landsat 8 C2 L2."""
    qa = image.select('QA_PIXEL')
    cloud_shadow = 1 << 4
    cloud = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(qa.bitwiseAnd(cloud).eq(0))
    return image.updateMask(mask)

## Diagnóstico: verificar disponibilidad de imágenes por año

In [20]:
for y in range(2016, 2026):
    count_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(f"{y}-01-01", f"{y}-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .size().getInfo()
    )
    count_l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_muni_geom)
        .filterDate(f"{y}-01-01", f"{y}-12-31")
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .size().getInfo()
    )
    print(y, "S2:", count_s2, "| L8:", count_l8)

C:\Users\Leo\miniconda3\envs\Geoprocesamiento\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2016 S2: 0 | L8: 1
2017 S2: 2 | L8: 3
2018 S2: 2 | L8: 0
2019 S2: 30 | L8: 3
2020 S2: 28 | L8: 2
2021 S2: 23 | L8: 1
2022 S2: 7 | L8: 1
2023 S2: 26 | L8: 2
2024 S2: 25 | L8: 2
2025 S2: 15 | L8: 0


### Esto demuestra dos problemas 
1. Sentinel-2: solo falla 2016 (0 imágenes). Los demás años tienen suficientes.
2. Landsat 8: está muy escaso en casi todos los años (0 a 3 imágenes), y falla por completo en 2018 y 2025.

In [21]:
# Sentinel-2: excluye 2016 (sin imágenes)
S2_YEARS = list(range(2017, 2026))
S2_CLOUD_THRESHOLD = 30

# Landsat 8: excluye años sin imágenes (ajusta según lo que confirmes con el diagnóstico),
# y usa un umbral de nubosidad más permisivo por su baja frecuencia de paso
L8_YEARS = list(range(2019, 2025))  # ejemplo, ajusta según tu diagnóstico
L8_CLOUD_THRESHOLD = 40

# Escoger indice multiespectral 
### El NDWI (Normalized Difference Water Index) es un índice espectral utilizado para identificar y delimitar cuerpos y superficies de agua, aprovechando la diferencia en la respuesta de la vegetación, el suelo y el agua en determinadas bandas del espectro electromagnético.

### aca se presenta la funcion primero para Sentinel 2 y luego para Landsat 8 

In [22]:
def compute_annual_index_s2(year, geom):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_CLOUD_THRESHOLD))
        .map(mask_s2_clouds_scl)
        .median()
    )
    index_img = s2.normalizedDifference(["B3", "B8"]).rename(f"NDWI_{year}")
    return index_img.clip(geom)


def compute_annual_index_l8(year, geom):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    def scale_l8(img):
        optical = img.select('SR_B.').multiply(0.0000275).add(-0.2)
        return img.addBands(optical, None, True)
    l8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUD_COVER", L8_CLOUD_THRESHOLD))
        .map(mask_l8_clouds_qa)
        .map(scale_l8)
        .median()
    )
    index_img = l8.normalizedDifference(["SR_B3", "SR_B5"]).rename(f"NDWI_{year}")
    return index_img.clip(geom)

## Definir rangos de busqueda de las escenas para cada uno de los dos sensores
### Esto es con vase a la verificaciones anteriores, en donde se detecto que cada sensor tenia datos en los años

In [30]:
S2_YEARS = list(range(2017, 2024))
S2_CLOUD_THRESHOLD = 30

L8_YEARS = list(range(2017, 2024))  
L8_CLOUD_THRESHOLD = 40

# Se escogió este rango de años pues son en donde se tiene información para los dos sensores

# Genera la lista de imágenes anuales para cada sensor

In [31]:
s2_images = [compute_annual_index_s2(y, ee_muni_geom) for y in S2_YEARS]
l8_images = [compute_annual_index_l8(y, ee_muni_geom) for y in L8_YEARS]

## Construir Stacks de imagenes

In [32]:
# Stack de Sentinel-2 (una banda por año, ya renombradas dentro de la función)
s2_stack = ee.Image.cat(s2_images)
print("Bandas S2 stack:", s2_stack.bandNames().getInfo())

# Stack de Landsat 8
l8_stack = ee.Image.cat(l8_images)
print("Bandas L8 stack:", l8_stack.bandNames().getInfo())

Bandas S2 stack: ['NDWI_2017', 'NDWI_2018', 'NDWI_2019', 'NDWI_2020', 'NDWI_2021', 'NDWI_2022', 'NDWI_2023']
Bandas L8 stack: ['NDWI_2017', 'NDWI_2018', 'NDWI_2019', 'NDWI_2020', 'NDWI_2021', 'NDWI_2022', 'NDWI_2023']


## Verificación del stecks 

In [33]:
print("N° bandas S2:", len(s2_stack.bandNames().getInfo()))
print("N° bandas L8:", len(l8_stack.bandNames().getInfo()))

N° bandas S2: 7
N° bandas L8: 7


# Exportar

In [34]:
# Exportar stack de Sentinel-2
task_s2 = ee.batch.Export.image.toDrive(
    image=s2_stack,
    description='NDWI_Sentinel2_2017_2025_Sabaneta',
    folder='Resultados1',
    fileNamePrefix='NDWI_S2_Sabaneta_2017_2025',
    region=ee_muni_geom,
    scale=10,
    crs='EPSG:9377',
    maxPixels=1e9
)
task_s2.start()
print("Exportación de Sentinel-2 iniciada.")

# Exportar stack de Landsat 8
task_l8 = ee.batch.Export.image.toDrive(
    image=l8_stack,
    description='NDWI_Landsat8_2019_2024_Sabaneta',
    folder='Resultados1',
    fileNamePrefix='NDWI_L8_Sabaneta_2019_2024',
    region=ee_muni_geom,
    scale=30,
    crs='EPSG:9377',
    maxPixels=1e9
)
task_l8.start()
print("Exportación de Landsat 8 iniciada.")

Exportación de Sentinel-2 iniciada.
Exportación de Landsat 8 iniciada.


## Verificación 

In [36]:
print(task_s2.status())
print(task_l8.status())

{'state': 'READY', 'description': 'NDWI_Sentinel2_2017_2025_Sabaneta', 'priority': 100, 'creation_timestamp_ms': 1788991583664, 'update_timestamp_ms': 1788991595969, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'XWK2FRSD54YL36OMMC243NSQ', 'name': 'projects/312893530331/operations/XWK2FRSD54YL36OMMC243NSQ'}
{'state': 'READY', 'description': 'NDWI_Landsat8_2019_2024_Sabaneta', 'priority': 100, 'creation_timestamp_ms': 1788991584319, 'update_timestamp_ms': 1788991584319, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'Z3XAIWPBSYQKBHGKDYFLC4ZC', 'name': 'projects/312893530331/operations/Z3XAIWPBSYQKBHGKDYFLC4ZC'}
